# TunedAI — Causal Reasoning: 4-Model Comparison

This notebook compares four AI models on the same causal reasoning questions:

| Model | What it is |
|---|---|
| Base Qwen 2.5-7B | An unmodified open-source model |
| GPT-4o | OpenAI's best model (optional — needs API key) |
| Claude 3.5 Sonnet | Anthropic's best model (optional — needs API key) |
| **TunedAI Causal Model** | The same Qwen 2.5-7B, fine-tuned by TunedAI on causal reasoning |

---

## Before you start — one required step

**You must enable a free GPU or the models will load very slowly.**

1. Click **Runtime** in the menu above
2. Click **Change runtime type**
3. Under **Hardware accelerator**, select **T4 GPU**
4. Click **Save**

Then click **Runtime → Run all** (or press Ctrl+F9). The notebook will run automatically.

Loading takes about **2 minutes**. You will see progress messages appear. Scroll down after it finishes to read the results.

---

### About the questions

Every question in this notebook comes from a book or paper published **before AI existed**. The correct answers were worked out by human experts — philosophers, doctors, and statisticians — decades ago. This means the models cannot have simply memorized the answers from the internet.

Sources: John Snow (1854), Ignaz Semmelweis (1847), E.H. Simpson (1951), Bradford Hill (1965), David Hume (1748), R.A. Fisher (1935)

---
## Cell 1 of 4 — Install required packages

This installs the software needed to run the models. You will see a lot of text scroll by — that is normal. Wait for it to finish before the next cell runs.

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes torch openai anthropic

---
## Cell 2 of 4 — Optional: Add your API keys

If you want to include **GPT-4o** and **Claude 3.5** in the comparison, paste your API keys below between the quotes.

- OpenAI key: get one at **platform.openai.com** → API Keys
- Anthropic key: get one at **console.anthropic.com** → API Keys

**If you leave these blank, those columns will be skipped.** Base Qwen vs TunedAI still runs with no keys needed.

In [ ]:
OPENAI_API_KEY    = ""   # paste your OpenAI key here (optional)
ANTHROPIC_API_KEY = ""   # paste your Anthropic key here (optional)

---
## Cell 3 of 4 — Load the models

This downloads and loads two versions of the same AI model:
- The original unmodified version (Base Qwen 2.5-7B)
- TunedAI's fine-tuned version with causal reasoning training

**This takes about 2 minutes.** You will see messages like "Loading tokenizer", "Loading base model", "Loading TunedAI adapter". Wait for the green checkmark ✓ before continuing.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import openai, anthropic

BASE_MODEL   = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER_REPO = "tunedailabs/causal-reasoning-qwen-7b"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

print("Loading base model (largest step — about 90 seconds)...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("Loading TunedAI causal reasoning adapter...")
tuned_model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
tuned_model.eval()

oai_client = openai.OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
ant_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None

print("\n✓ All models ready. Scroll down to see the results.")

---
## Cell 4 of 4 — Helper functions

This sets up the comparison engine. It runs automatically — nothing to change here.

In [ ]:
SYSTEM = "You are a careful reasoner. Answer questions about causation, association, intervention, and counterfactuals precisely and correctly."

def ask_local(question, use_adapter=True, max_new_tokens=350):
    if use_adapter:
        tuned_model.enable_adapter_layers()
    else:
        tuned_model.disable_adapter_layers()
    messages = [{"role":"system","content":SYSTEM},{"role":"user","content":question}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(tuned_model.device)
    with torch.no_grad():
        out = tuned_model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.1, do_sample=False)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

def ask_gpt4(question):
    if not oai_client:
        return "[Skipped — no OpenAI API key provided]"
    r = oai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role":"system","content":SYSTEM},{"role":"user","content":question}],
        max_tokens=400
    )
    return r.choices[0].message.content.strip()

def ask_claude(question):
    if not ant_client:
        return "[Skipped — no Anthropic API key provided]"
    r = ant_client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=400,
        system=SYSTEM,
        messages=[{"role":"user","content":question}]
    )
    return r.content[0].text.strip()

def compare_all(question, source="", note=""):
    SEP = "=" * 70
    DIV = "-" * 70
    print(SEP)
    if source: print(f"SOURCE : {source}")
    if note:   print(f"NOTE   : {note}")
    print(SEP)
    print(f"QUESTION:\n{question}\n")
    for label, fn in [
        ("BASE QWEN 2.5-7B (untuned — the starting point)", lambda q: ask_local(q, use_adapter=False)),
        ("GPT-4o",                                          ask_gpt4),
        ("CLAUDE 3.5 SONNET",                               ask_claude),
        ("TUNEDAI CAUSAL MODEL ★",                          lambda q: ask_local(q, use_adapter=True)),
    ]:
        print(DIV)
        print(f"[ {label} ]")
        print(DIV)
        print(fn(question))
        print()

print("✓ Ready — running all tests now...")

---
# Results

Each section below shows all four models answering the same question.

The **BASE QWEN** column is the model before any training on causal reasoning.
The **TUNEDAI** column is the same model after fine-tuning.

---
## Test 1 — Simpson's Paradox
**Source:** E.H. Simpson, *The Interpretation of Interaction in Contingency Tables*, 1951

A real treatment study where one treatment is better for every subgroup of patients — but appears worse in the combined total. Choosing the wrong treatment could harm patients. The correct answer requires understanding why the combined statistic is misleading.

In [ ]:
compare_all(
    question="""Kidney stone treatment study:

Small stones: Treatment A 93% success (81/87 patients), Treatment B 87% success (234/270 patients)
Large stones:  Treatment A 73% success (192/263 patients), Treatment B 69% success (55/80 patients)
Combined:      Treatment A 78% (273/350), Treatment B 83% (289/350)

Treatment A is better for patients with small stones.
Treatment A is better for patients with large stones.
Yet Treatment B appears better in the combined total.

A patient arrives and the doctor does not yet know their stone size.
Which treatment should the doctor give, and why does the combined percentage give the wrong answer?""",
    source="E.H. Simpson, 1951",
    note="Correct answer: administer Treatment A. The combined total is misleading because stone size is a confounder."
)

---
## Test 2 — John Snow and the Cholera Pump
**Source:** John Snow, *On the Mode of Communication of Cholera*, 1854

Snow noticed that households using one water company had far higher cholera death rates. Then he removed the handle from the Broad Street pump and the local outbreak stopped. These are two different types of evidence — and only one of them justifies a causal claim.

In [ ]:
compare_all(
    question="""In 1854, John Snow observed:
- Households using Southwark & Vauxhall water: 315 cholera deaths per 10,000 houses
- Households using Lambeth water: 37 deaths per 10,000 houses

He then removed the handle from the Broad Street water pump.
The cholera outbreak in that district subsided.

Question 1: What does the difference in death rates between the two water companies tell us?
Question 2: What does the pump handle removal tell us that the death rate comparison alone cannot?
Question 3: Why does the order of reasoning matter — starting from a mechanism and then intervening, rather than just observing patterns?""",
    source="John Snow, 1854",
    note="Correct answer: the comparison establishes association. The pump removal establishes causation — it is a real-world intervention."
)

---
## Test 3 — Semmelweis and Handwashing
**Source:** Ignaz Semmelweis, Vienna General Hospital records, 1847

Semmelweis noticed that the ward staffed by doctors (who performed autopsies) had dramatically higher death rates than the ward staffed by midwives. He then mandated handwashing and death rates collapsed. This is a before-and-after intervention.

In [ ]:
compare_all(
    question="""In 1847, Semmelweis observed at Vienna General Hospital:
- Doctors' ward (doctors went from performing autopsies to delivering babies): 10-35% death rate from childbed fever
- Midwives' ward (midwives did not perform autopsies): 1-2% death rate

Semmelweis then required all doctors to wash their hands with chlorinated lime solution before entering the maternity ward.
The death rate in the doctors' ward immediately fell below 2%.

Before the handwashing requirement: could Semmelweis conclude that performing autopsies caused the deaths?
After the handwashing requirement: what can he now conclude that he could not before?
Explain the difference between what the observation tells us and what the intervention tells us.""",
    source="Ignaz Semmelweis, 1847",
    note="Correct answer: observation establishes association. Intervention (handwashing mandate) establishes that the transmission pathway could be broken — a causal claim."
)

---
## Test 4 — Bradford Hill and Smoking
**Source:** Bradford Hill, *The Environment and Disease: Association or Causation?*, 1965

No experiment was ever run on smoking — you cannot randomly assign people to smoke for decades. Critics argued that without an experiment, you can only say smoking is correlated with lung cancer, not that it causes it. Bradford Hill proposed a way to reason about causation from observation alone.

In [ ]:
compare_all(
    question="""By 1965, scientists had observed:
- Smokers get lung cancer at 9-10 times the rate of non-smokers
- This finding had been replicated across many countries and decades
- Heavier smokers get cancer at higher rates than lighter smokers
- Lung cancer is diagnosed years after someone starts smoking, never before
- Cigarette smoke contains known carcinogens

No randomized controlled trial was conducted. You cannot force people to smoke for decades.
Critics argued: without an experiment, this is only a correlation.

Can we conclude that smoking causes lung cancer from this evidence?
What is the strongest argument that we can?
What is the strongest argument that we cannot?""",
    source="Bradford Hill, 1965",
    note="Correct answer: yes, the totality of evidence crosses the threshold for causal inference. Temporality and dose-response are particularly powerful."
)

---
## Test 5 — Hume's Billiard Balls (and Where the Logic Breaks)
**Source:** David Hume, *An Enquiry Concerning Human Understanding*, 1748

Hume proposed a simple test for causation: if the first thing had not happened, the second thing would not have happened. This works for simple cases but breaks down in a specific type of situation. Philosophers spent 250 years trying to fix it.

In [ ]:
compare_all(
    question="""Hume (1748) proposed: to find the cause of something, ask — if the first event had NOT happened, would the second event still have happened? If not, the first event is the cause.

Apply this test to two cases:

Case 1 — Simple: A driver runs a red light and strikes a pedestrian who is injured.
If the driver had NOT run the red light, would the pedestrian have been injured? No.
So by Hume's test: the driver running the red light caused the injury. ✓

Case 2 — Harder: Two people independently decide to poison the same victim's drink.
Person A adds a lethal dose of poison. Person B (not knowing this) also adds a lethal dose.
The victim drinks and dies.

Apply Hume's test:
- If Person A had NOT added poison: the victim still dies (from B's poison). So A did NOT cause the death?
- If Person B had NOT added poison: the victim still dies (from A's poison). So B did NOT cause the death?

Does Hume's test give the right answer for Case 2? If not, what does this reveal about the test's limits?""",
    source="David Hume, 1748",
    note="Correct answer: Hume's test fails for Case 2. Both people are causes, but neither passes the but-for test. This is the problem of overdetermination."
)

---
## Test 6 — Fisher and Randomization
**Source:** R.A. Fisher, *The Design of Experiments*, 1935

The same drug can appear harmful in one study and beneficial in another. Fisher showed that random assignment is what makes the difference — and explained exactly why.

In [ ]:
compare_all(
    question="""Two studies of the exact same drug:

Study A — Observational: Doctors choose which patients get the drug.
Doctors naturally give the drug to sicker patients (they need it more).
Result: The drug group has WORSE outcomes than the no-drug group.

Study B — Randomized (Fisher's method): A coin flip decides who gets the drug.
Result: The drug group has BETTER outcomes than the no-drug group.

Why do these studies give opposite results for the same drug?
What does random assignment do that doctor assignment cannot?
Why does Study B let us say 'the drug works' but Study A does not — even if Study A had 100 times more patients?""",
    source="R.A. Fisher, 1935",
    note="Correct answer: doctor assignment confounds sickness with treatment. Random assignment makes the two groups identical on all factors, so any difference must be due to the drug."
)

---
## Try Your Own Question

Replace the text below with any causal reasoning question. Examples to try:
- "Does wearing a seatbelt cause people to drive more recklessly?"
- "If a rooster crows before sunrise every day, does the rooster cause the sun to rise?"
- "A city adds more police and crime goes up. Does adding police cause more crime?"

In [ ]:
MY_QUESTION = """
Type your question here and run this cell.
"""

compare_all(MY_QUESTION, source="Custom")

---
## Share What You Saw

Open a [GitHub Issue](https://github.com/mgentry11/tunedai/issues/new) and paste your results.

Tell us which questions the TunedAI model got right that the others did not — or anything surprising you found.

We read every submission.

---

**TunedAI** — We fine-tune open-source LLMs for real-world reasoning.

[tunedailabs.com](https://tunedailabs.com)